# Apollo 15 & 17 HFE — Supporting Information (SI Figs S1–S6)

Companion notebook to `01_apollo_validation.ipynb` (the GRL letter main text). This notebook re-runs the same Hayne-2017 reference and Discrete-3-layer alternative solvers at both Apollo sites and produces the six supplementary figures referenced in §10 of the main notebook:

| SI ID | Figure |
|---|---|
| **S1** | Per-sensor diurnal grid — A15 |
| **S2** | Per-sensor diurnal grid — A17 |
| **S3** | Apollo observed vs modelled scatter (1:1 diagnostic) |
| **S4** | Residual vs depth (model − obs) |
| **S5** | Diurnal amplitude vs depth — borestem signature |
| **S6** | Spin-up convergence diagnostic |

The setup (bootstrap → solver runs → SPICE LST) is identical to the main notebook so this file is fully self-contained and can be Run-All in isolation.


## Bootstrap (click Run All — no setup required)

Auto-installs packages and downloads Apollo 15/17 PDS data on first run. Subsequent runs are cached.

In [ ]:
# === Lunar-V2 bootstrap — safe to re-run =================================
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
for _p in (_here, *_here.parents):
    if (_p / 'pyproject.toml').is_file() and (_p / 'lunar' / '_bootstrap.py').is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
else:
    raise RuntimeError('Could not find Lunar-V2 repo root from ' + str(_here))

from lunar import _bootstrap as boot
boot.ensure_lunar(extra=('spiceypy', 'scipy'))
boot.ensure_apollo_hfe(mission='a15',
                       probes=('p1f1', 'p1f2', 'p1f3', 'p1f4',
                               'p2f1', 'p2f2', 'p2f3', 'p2f4'))
boot.ensure_apollo_hfe(mission='a17', probes=())
boot.ensure_spice_kernels()


In [ ]:
from __future__ import annotations
import sys, pathlib, os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines  import Line2D
from matplotlib.patches import Patch

from lunar.validation import load_apollo_hfe_temperature, load_apollo_hfe_depth
from lunar.grid import make_geometric_grid
from lunar.properties import conductivity_hayne, density_hayne, specific_heat
from lunar.constants import (
    SIGMA_SB, EMISSIVITY_DEFAULT, CHI_RADIATIVE, T_REFERENCE,
    K_SURFACE, K_DEEP, H_PARAMETER, LUNATION_SECONDS,
)
from lunar.solver import PixelInputs, solve_pixel
from lunar.apollo_helpers import (
    extract_sensor_stability, print_stability_table,
    iso_to_seconds, find_stable_window, run_site_solvers,
    compute_validation_stats,
)
from lunar.plotting.style_guide import (
    apply_style, COLORS, save_figure, panel_label,
    FIG_SINGLE, FIG_DOUBLE, FIG_DOUBLE_TALL,
)

apply_style()

REPO_ROOT  = str(_p)
OUT_DIR    = os.path.join(REPO_ROOT, 'output', 'figures')
os.makedirs(OUT_DIR, exist_ok=True)
print('Imports OK. Figures →', OUT_DIR)


## §1  Site + model parameters

All tuneable values in one place. Deep references in comments.

In [ ]:
# ── Physical constants ──────────────────────────────────────────────────
S0           = 1361.0          # W m⁻² — solar constant (Kopp & Lean 2011, GRL 38 L01706)
T_LUNAR      = LUNATION_SECONDS  # 29.530589 d synodic period [s]
DT_STEP      = 3600.0          # s — time step (1 h)
N_LUNATIONS  = 100             # spin-up lunations (≥80 for deep convergence)
SPINUP_TOL   = 0.01            # K — convergence threshold

# ── Depth grid ───────────────────────────────────────────────────────────
GRID = dict(z_max=5.0, dz0=0.002, growth=0.08)

# ── Apollo site parameters ───────────────────────────────────────────────
# Coordinates: NASA ALSEP gazetteer / LROC NAC seismometer locations.
# Albedo:       Vasavada et al. (2012) Icarus 218, 558 — Diviner-derived
#               broadband Bond albedo at the Apollo lat band
#               (A15 mare/Imbrium = 0.131; A17 Taurus-Littrow = 0.137).
# Emissivity:   Bandfield et al. (2015) Icarus 248, 357 — broadband ε
#               for mare and highland regolith (≈ 0.95 ± 0.02).
# Q_BASAL (nominal): published HFE surface heat flow values.
#               A15 = 21 mW m⁻² (Langseth et al. 1976), A17 = 15 mW m⁻²
#               (Nagihara et al. 2018).  Note the Saito et al. (2007)
#               and Nagihara et al. (2018) reanalyses suggest the
#               original Langseth A15 value is biased high; the
#               reprocessed value is closer to ≈ 14 mW m⁻².  A
#               Q_b sensitivity sweep is run in §7.6 to bracket this.
# T_MEAN_EFF:   only the steady-state initial guess (irrelevant after
#               spin-up); chosen to match Diviner annual mean for the lat.
SITES = {
    'A15': dict(
        label='Apollo 15',  lat=26.13, lon=3.63,
        albedo=0.131,                     # Vasavada+2012 mare value
        albedo_source='Vasavada+2012',
        emissivity=0.95,                  # Bandfield+2015
        emissivity_source='Bandfield+2015',
        Q_BASAL=0.021,                    # 21 mW m⁻²  Langseth 1976 (nominal)
        Q_BASAL_alt=0.014,                # 14 mW m⁻²  Saito 2007 / Nagihara 2018 reanalysis
        Q_BASAL_source='Langseth+1976 (alt: Saito+2007, Nagihara+2018)',
        T_MEAN_EFF=250.0,
        MIN_DEPTH_CM=80,
        y_lim=160,
        mission='a15',
        T_diviner_max_K=380.0,            # Vasavada+2012 Fig 7 lat-band typical max
        T_diviner_min_K=92.0,             # Vasavada+2012 Fig 7 lat-band typical min
        T_diviner_source='Vasavada+2012 Fig 7 (Diviner 26°N mean)',
    ),
    'A17': dict(
        label='Apollo 17',  lat=20.19, lon=30.77,
        albedo=0.137,                     # Vasavada+2012 highland value
        albedo_source='Vasavada+2012',
        emissivity=0.95,                  # Bandfield+2015
        emissivity_source='Bandfield+2015',
        Q_BASAL=0.015,                    # 15 mW m⁻²  Nagihara 2018 (reprocessed)
        Q_BASAL_alt=0.018,                # 18 mW m⁻²  original Langseth 1976
        Q_BASAL_source='Nagihara+2018 (alt: Langseth+1976 original)',
        T_MEAN_EFF=255.0,
        MIN_DEPTH_CM=80,
        y_lim=240,
        mission='a17',
        T_diviner_max_K=387.0,            # Vasavada+2012 Fig 7 lat-band typical max
        T_diviner_min_K=95.0,
        T_diviner_source='Vasavada+2012 Fig 7 (Diviner 20°N mean)',
    ),
}

# ── Hayne (2017) model — from constants.py, confirmed vs App. A ──────────
HAYNE = dict(
    K_SURFACE=K_SURFACE,   # 7.4e-4 W m⁻¹ K⁻¹   Table 2
    K_DEEP=K_DEEP,         # 3.4e-3 W m⁻¹ K⁻¹   Table 2
    H_PARAM=H_PARAMETER,   # 0.06 m               Table 2
    CHI=CHI_RADIATIVE,     # 2.7                  Table 2
    T_REF=T_REFERENCE,     # 350 K                App. A
)

# ── Discrete 3-layer model ──────────────────────────────────────────────
# PROVENANCE: this 3-layer parameterisation is *not* a fit to the
# Apollo subsurface T_eq.  Layer thicknesses (H1=7 cm, H2=20 cm) and
# bulk-density anchors (rho_s=1100, rho_d=1700, rho_max=1800 kg m⁻³)
# are taken directly from the Hayne (2017) Appendix A profile evaluated
# at three discrete depths — i.e. a stair-step approximation of the
# exponential.  The (K_SOLID_SURF, K_SOLID_DEEP) pair was inherited from
# the lunar1Dheat reference solver (Martinez & Siegler 2021) and chosen
# to keep K_d within 2× of the Hayne value for the deep layer, while
# giving the shallow layer slightly higher contact conductivity to
# match Apollo TR drill data (Langseth+1976 shallow gradient).
# In §7.5 the Discrete model is therefore a *physically-motivated
# alternative parameterisation*, not a free-knob calibration; both
# sites are run with the same parameter set.
DISCRETE = dict(
    H_LAYER1=0.07,         # m  top fluffy layer
    H_LAYER2=0.20,         # m  end of transition
    K_SOLID_SURF=1.0e-3,   # W m⁻¹ K⁻¹
    K_SOLID_DEEP=6.3e-3,   # W m⁻¹ K⁻¹
    RHO_SURF=1100.0,
    RHO_DEEP=1700.0,
    RHO_MAX=1800.0,
    RHO_EFOLD=0.5,         # m
    CHI=CHI_RADIATIVE,
    T_REF=T_REFERENCE,
)

# ── Colours ──────────────────────────────────────────────────────────────
CLR_HAYNE    = '#2471A3'   # blue   — Hayne 2017
CLR_DISC     = '#C0392B'   # red    — Discrete 3-layer
CLR_TG       = '#0B1D51'   # navy   — TG sensor
CLR_TR       = '#7A1B1B'   # maroon — TR sensor
ZONE_CLR     = '#E8DAEF'   # lavender — diurnal exclusion shading
ZONE_LINE    = '#7D3C98'   # purple   — exclusion boundary

# Build shared grid and time array once
grid   = make_geometric_grid(**GRID)
z_mid  = grid.z_mid
z_cm   = z_mid * 100.0
N_t    = int(T_LUNAR / DT_STEP) + 1
t_s    = np.linspace(0.0, T_LUNAR, N_t)

print(f'Grid: {grid.n_layers} layers, z_max = {z_mid[-1]:.2f} m')
print(f'Time: {N_t} steps / lunation')
print()
print('Per-site parameter provenance:')
for tag, cfg in SITES.items():
    print(f'  {cfg["label"]:<11} '
          f'α={cfg["albedo"]:.3f} ({cfg["albedo_source"]}) '
          f'ε={cfg["emissivity"]:.2f} ({cfg["emissivity_source"]}) '
          f'Q_b={cfg["Q_BASAL"]*1e3:.0f} mW/m² ({cfg["Q_BASAL_source"]})')


## §2  Load Apollo HFE data

Stability windows extracted with |dT/dt| ≤ 0.08 K yr⁻¹ criterion on the tail of each sensor record.

In [ ]:
# ── Load stability windows for both sites ──────────────────────────────
hfe = {}
for tag, cfg in SITES.items():
    print(f'\n── {cfg["label"]} ──')
    bundle = extract_sensor_stability(cfg['mission'], cfg['MIN_DEPTH_CM'])
    hfe[tag] = bundle
    print_stability_table(bundle, cfg['label'], cfg['MIN_DEPTH_CM'])


## §4  Run thermal solvers

Both models run at each site (100-lunation spin-up). Only K(T,z) and rho(z) differ.

In [ ]:
# Discrete 3-layer property functions
def k_discrete(T, z):
    H1, H2 = DISCRETE['H_LAYER1'], DISCRETE['H_LAYER2']
    Ks, Kd = DISCRETE['K_SOLID_SURF'], DISCRETE['K_SOLID_DEEP']
    chi, T_ref = DISCRETE['CHI'], DISCRETE['T_REF']
    z = np.asarray(z, dtype=float); T = np.asarray(T, dtype=float)
    ks = np.where(z < H1, Ks,
         np.where(z < H2, Ks + (Kd-Ks)*(z-H1)/(H2-H1), Kd))
    return ks * (1.0 + chi*(T/T_ref)**3)

def rho_discrete(z):
    H1, H2 = DISCRETE['H_LAYER1'], DISCRETE['H_LAYER2']
    rs, rd, rm, re = (DISCRETE['RHO_SURF'], DISCRETE['RHO_DEEP'],
                      DISCRETE['RHO_MAX'],  DISCRETE['RHO_EFOLD'])
    z = np.asarray(z, dtype=float)
    return np.where(z < H1, rs,
           np.where(z < H2, rs + (rd-rs)*(z-H1)/(H2-H1),
               rd + (rm-rd)*(1 - np.exp(-(z-H2)/re))))

print('Discrete model functions defined.')


In [ ]:
# Run Hayne + Discrete solvers for both sites
runs  = {}
stats = {}

for tag, cfg in SITES.items():
    print(f'\n---- {cfg["label"]} (lat={cfg["lat"]}N, Q_b={cfg["Q_BASAL"]*1e3:.0f} mW/m2) ----')
    run = run_site_solvers(
        cfg, grid, t_s, HAYNE,
        K_func_hayne=conductivity_hayne,
        K_func_disc=k_discrete,
        rho_func_disc=rho_discrete,
        cp_func=specific_heat,
        s0_nominal=S0, sun_scale=1.0,
        t_lunar=T_LUNAR, n_lunations=N_LUNATIONS,
        spinup_tol=SPINUP_TOL,
    )
    st = compute_validation_stats(hfe[tag], run, z_cm)
    runs[tag]  = run
    stats[tag] = st
    n_d = hfe[tag]['deep_mask'].sum()
    d   = cfg['MIN_DEPTH_CM']
    print(f'  Hayne    RMSE(z>={d}cm, N={n_d}) = {st["rmse_hayne"]:.3f} K  bias = {st["bias_hayne"]:+.3f} K')
    print(f'  Discrete RMSE(z>={d}cm, N={n_d}) = {st["rmse_disc"]:.3f} K  bias = {st["bias_disc"]:+.3f} K')

print('Both sites done.')


In [ ]:
# Build SPICE LST lookup table (once, shared by both sites)
from lunar.ephem import _furnish_kernels as _furnish_spice
import spiceypy as _spice
from datetime import datetime, timezone

_furnish_spice()

T_SYN_HR = 29.530589 * 24.0   # 708.734 h

def _iso_to_unix(s):
    s2 = s.rstrip('Z') + '+00:00' if s.endswith('Z') else s
    return datetime.fromisoformat(s2).replace(tzinfo=timezone.utc).timestamp()

def _unix_to_et(arr):
    return np.array([_spice.unitim(u/86400.0 + 2440587.5, 'JED', 'ET')
                     for u in arr], dtype=np.float64)

_N_GRID    = int(42 * 365.25)
_unix_grid = np.linspace(_iso_to_unix('1970-01-01T00:00:00'),
                         _iso_to_unix('2012-01-01T00:00:00'), _N_GRID)
_et_grid   = _unix_to_et(_unix_grid)

_sub_lon_raw = np.empty(_N_GRID)
for _i, _et in enumerate(_et_grid):
    _pos, _ = _spice.spkpos('SUN', float(_et), 'MOON_ME', 'LT+S', 'MOON')
    _sub_lon_raw[_i] = np.rad2deg(np.arctan2(_pos[1], _pos[0]))
_sub_lon_unwrap = np.unwrap(np.deg2rad(_sub_lon_raw))

def spice_lst(t_unix, site_lon):
    t    = np.atleast_1d(np.asarray(t_unix, dtype=float))
    slon = np.rad2deg(np.interp(t, _unix_grid, _sub_lon_unwrap)) % 360.0
    HA   = ((site_lon - slon + 180.0) % 360.0) - 180.0
    return (12.0 + HA / 15.0) % 24.0

def lst_to_lunhour(lst_hr):
    return np.asarray(lst_hr, float) * (T_SYN_HR / 24.0)

def _bin_med_iqr(lst, T, edges):
    nb = len(edges) - 1
    med = np.full(nb, np.nan); q25 = med.copy(); q75 = med.copy()
    for k in range(nb):
        sel = (lst >= edges[k]) & (lst < edges[k+1])
        if sel.sum() >= 3:
            med[k] = np.median(T[sel])
            q25[k] = np.percentile(T[sel], 25)
            q75[k] = np.percentile(T[sel], 75)
    return med, q25, q75

def _pi(lst_q, lst_m, T_m):
    x = np.asarray(lst_m, float) % 24.0
    y = np.asarray(T_m, float)
    s = np.argsort(x)
    x2 = np.r_[x[s], x[s]+24.0]; y2 = np.r_[y[s], y[s]]
    return np.interp(np.asarray(lst_q, float) % 24.0, x2, y2)

N_BINS    = 72
BIN_EDGES = np.linspace(0, 24, N_BINS + 1)
BIN_CTR   = 0.5 * (BIN_EDGES[:-1] + BIN_EDGES[1:])
BIN_LUNHR = lst_to_lunhour(BIN_CTR)

# Model LST mapping (t=0 = local noon for sinusoidal forcing)
_t_int    = t_s[:-1]
LST_MOD   = ((_t_int - T_LUNAR/2.0) % T_LUNAR) / T_LUNAR * 24.0

H_SUNRISE = T_SYN_HR / 4.0
H_NOON    = T_SYN_HR / 2.0
H_SUNSET  = 3.0 * T_SYN_HR / 4.0

print(f'SPICE LST table: {_N_GRID} daily points (1970-2012)')


In [ ]:
# Figs 3 & 4: per-sensor diurnal grid for A15 and A17
# -------------------------------------------------------
CLR_H_D   = '#E67E22'   # orange  — Hayne diurnal
CLR_D_D   = '#1565C0'   # blue    — Discrete diurnal
ZONE_EXCL = '#FADBD8'; ZONE_VAL = '#D4EFDF'; NIGHT_G = '#ececec'

def _diurnal_site_figure(tag, figname):
    cfg      = SITES[tag]
    bundle   = hfe[tag]
    out_h    = runs[tag]['out_hayne']
    out_d    = runs[tag]['out_disc']
    site_lon = cfg['lon']
    min_d    = cfg['MIN_DEPTH_CM']

    obs_all = {}
    for dtab in [bundle['d1'], bundle['d2']]:
        for sensor in np.unique(dtab['sensor']):
            mask  = dtab['sensor'] == sensor
            t_u   = iso_to_seconds(dtab['time_iso'][mask])
            T_obs = dtab['T'][mask].astype(float)
            good  = np.isfinite(T_obs) & (T_obs > 50.0)
            t_u, T_obs = t_u[good], T_obs[good]
            if len(t_u) < 10:
                continue
            lst = spice_lst(t_u, site_lon)
            key = sensor.strip()
            nh = len(lst) // 2
            obs_all[key] = (lst[nh:], T_obs[nh:])

    sorted_sens = sorted(bundle['sensors'], key=lambda s: s['depth_cm'])
    n_s = len(sorted_sens)
    NCOLS = 3
    NROWS = int(np.ceil(n_s / NCOLS))

    fig, axes = plt.subplots(NROWS, NCOLS,
                             figsize=(6.5*NCOLS, 3.8*NROWS),
                             constrained_layout=False)
    axes_flat = np.array(axes).reshape(-1)

    for idx, s in enumerate(sorted_sens):
        ax       = axes_flat[idx]
        sensor   = s['sensor']
        depth_cm = s['depth_cm']
        is_deep  = depth_cm >= min_d
        iz       = int(np.argmin(np.abs(z_cm - depth_cm)))

        ax.set_facecolor(ZONE_VAL if is_deep else ZONE_EXCL)
        ax.axvspan(0.0, H_SUNRISE, color=NIGHT_G, alpha=0.55, zorder=0)
        ax.axvspan(H_SUNSET, T_SYN_HR, color=NIGHT_G, alpha=0.55, zorder=0)
        for xv in (H_SUNRISE, H_NOON, H_SUNSET):
            ax.axvline(xv, color='#E67E22', ls=':', lw=1.0, alpha=0.7, zorder=1)

        _Th = out_h.T[iz, :-1]
        _Td = out_d.T[iz, :-1]
        mod_pk_h = float(LST_MOD[np.argmax(_Th)])
        mod_pk_d = float(LST_MOD[np.argmax(_Td)])

        _obs_pk = np.nan
        if sensor in obs_all:
            _lo, _To = obs_all[sensor]
            _mt, _, _ = _bin_med_iqr(_lo, _To, BIN_EDGES)
            _ok = np.isfinite(_mt)
            if _ok.sum() >= 5:
                _obs_pk = float(BIN_CTR[_ok][np.argmax(_mt[_ok])])

        if np.isfinite(_obs_pk):
            lag_h = (mod_pk_h - _obs_pk + 12.0) % 24.0 - 12.0
            lag_d = (mod_pk_d - _obs_pk + 12.0) % 24.0 - 12.0
        else:
            lag_h = lag_d = 0.0

        T_h_u = _pi(BIN_CTR, LST_MOD, _Th)
        T_d_u = _pi(BIN_CTR, LST_MOD, _Td)
        T_h_s = _pi((BIN_CTR + lag_h) % 24.0, LST_MOD, _Th)
        T_d_s = _pi((BIN_CTR + lag_d) % 24.0, LST_MOD, _Td)

        ax.plot(BIN_LUNHR, T_h_u, color=CLR_H_D, lw=1.2, ls='--', alpha=0.28, zorder=3)
        ax.plot(BIN_LUNHR, T_d_u, color=CLR_D_D, lw=1.2, ls='-',  alpha=0.28, zorder=3)
        ax.plot(BIN_LUNHR, T_h_s, color=CLR_H_D, lw=2.4, ls='--', alpha=1.00, zorder=4)
        ax.plot(BIN_LUNHR, T_d_s, color=CLR_D_D, lw=2.8, ls='-',  alpha=1.00, zorder=5)

        if sensor in obs_all:
            _lo, _To = obs_all[sensor]
            med, q25, q75 = _bin_med_iqr(_lo, _To, BIN_EDGES)
            ok = np.isfinite(med)
            if ok.any():
                stype = s.get('stype', sensor[:2])
                clr_o = CLR_TG if stype == 'TG' else CLR_TR
                mrk   = 'o'    if stype == 'TG' else 's'
                ax.fill_between(BIN_LUNHR[ok], q25[ok], q75[ok],
                                color=clr_o, alpha=0.18, zorder=2)
                ax.plot(BIN_LUNHR[ok], med[ok], marker=mrk, markersize=5,
                        ls='none', color=clr_o, zorder=6,
                        markeredgecolor='white', markeredgewidth=0.5)

        lbl_tag = '[valid.]' if is_deep else '[borestem]'
        t_clr   = '#1E8449' if is_deep else '#C0392B'
        lag_str = (f'  dH={lag_h:+.1f}h  dD={lag_d:+.1f}h'
                   if np.isfinite(_obs_pk) else '')
        ax.set_title(
            f'{sensor}   z = {depth_cm:.0f} cm   {lbl_tag}{lag_str}',
            fontsize=9.5, weight='bold', color=t_clr, pad=4
        )
        ax.set_xlim(0, T_SYN_HR)
        ax.set_xticks([0, H_SUNRISE, H_NOON, H_SUNSET, T_SYN_HR])
        ax.set_xticklabels(
            ['0 h', f'{H_SUNRISE:.0f} h', f'{H_NOON:.0f} h',
             f'{H_SUNSET:.0f} h', f'{T_SYN_HR:.0f} h'], fontsize=9)
        ax.tick_params(axis='y', labelsize=9)
        if idx % NCOLS == 0:
            ax.set_ylabel('T  [K]', fontsize=10)
        if idx // NCOLS == NROWS - 1:
            ax.set_xlabel('Elapsed lunar hours  (midnight = 0)', fontsize=10)

        _lo_y = min(T_h_u.min(), T_d_u.min(), T_h_s.min(), T_d_s.min())
        _hi_y = max(T_h_u.max(), T_d_u.max(), T_h_s.max(), T_d_s.max())
        if sensor in obs_all:
            _lo_obs, _To_obs = obs_all[sensor]
            _lo_y = min(_lo_y, float(np.percentile(_To_obs, 1)))
            _hi_y = max(_hi_y, float(np.percentile(_To_obs, 99)))
        pad = 0.18 * (_hi_y - _lo_y + 0.5)
        ax.set_ylim(_lo_y - pad, _hi_y + pad)
        ax.grid(True, alpha=0.2)

    for k in range(n_s, len(axes_flat)):
        axes_flat[k].set_visible(False)

    handles_leg = [
        Line2D([], [], marker='o', ls='none', color=CLR_TG, markersize=8,
               markeredgecolor='white', markeredgewidth=0.8,
               label='Apollo TG  (median \u00b1 IQR)'),
        Line2D([], [], marker='s', ls='none', color=CLR_TR, markersize=7,
               markeredgecolor='white', markeredgewidth=0.8,
               label='Apollo TR  (median \u00b1 IQR)'),
        Line2D([], [], color=CLR_H_D, lw=2.4, ls='--', label='Hayne 2017  (peak-aligned)'),
        Line2D([], [], color=CLR_D_D, lw=2.8, ls='-',  label='Discrete 3L  (peak-aligned)'),
        Line2D([], [], color=CLR_H_D, lw=1.2, ls='--', alpha=0.35, label='Hayne 2017  (unshifted)'),
        Line2D([], [], color=CLR_D_D, lw=1.2, ls='-',  alpha=0.35, label='Discrete 3L  (unshifted)'),
        Patch(facecolor=ZONE_VAL,  edgecolor='#1E8449', alpha=0.8,
              label=f'Validation zone  (z \u2265 {min_d} cm)'),
        Patch(facecolor=ZONE_EXCL, edgecolor='#C0392B', alpha=0.8,
              label='Borestem zone  (excluded from RMSE)'),
    ]
    fig.legend(handles=handles_leg, loc='lower center',
               bbox_to_anchor=(0.5, -0.01), ncol=4, fontsize=9.5,
               frameon=True, handlelength=2.2, columnspacing=1.6, borderpad=0.8)
    fig.suptitle(
        f'{cfg["label"]} HFE \u2014 Diurnal Cycle Comparison  (SPICE LST)\n'
        f'Peak-aligned model vs Apollo binned medians  (lat = {cfg["lat"]}\u00b0N)',
        fontsize=13, weight='bold'
    )
    fig.subplots_adjust(top=0.94, bottom=0.12, left=0.055, right=0.985,
                        hspace=0.58, wspace=0.24)
    save_figure(fig, figname, output_dir=OUT_DIR)
    plt.show()
    print(f'Saved: {figname}')

_diurnal_site_figure('A15', 'a15_diurnal_sensor_grid')
_diurnal_site_figure('A17', 'a17_diurnal_sensor_grid')


## §7  Validation statistics — tabulated summary

A two-site, two-model square. Deep-sensor RMSE, bias, MAE and R² are the
single-pixel goodness-of-fit metrics and Phase-1 success requires
RMSE ≤ 2 K at both sites for both models. We also export a LaTeX
``booktabs`` block ready for paste into the GRL letter.


In [ ]:
# §7  Tabulated validation statistics  (publication-ready)
# ──────────────────────────────────────────────────────────────────────
# Produces three artefacts:
#   1. Pretty in-notebook ASCII table (per-sensor and aggregate)
#   2. CSV file output/figures/apollo_validation_stats.csv
#   3. LaTeX booktabs block printed for direct paste into the manuscript

import csv, os, textwrap
from scipy.stats import pearsonr

# ---- aggregate table (RMSE / bias / MAE / R²) ------------------------------
agg_rows = []
for tag, cfg in SITES.items():
    st     = stats[tag]
    bundle = hfe[tag]
    n_deep = int(bundle['deep_mask'].sum())
    for mdl, key in (('Hayne 2017', 'hayne'), ('Discrete 3-layer', 'disc')):
        agg_rows.append({
            'site':  cfg['label'],
            'model': mdl,
            'N':     n_deep,
            'RMSE':  st[f'rmse_{key}'],
            'bias':  st[f'bias_{key}'],
            'MAE':   st[f'mae_{key}'],
            'R2':    st[f'r2_{key}'],
        })

print('=' * 78)
print('TABLE 1 — Apollo HFE deep-sensor validation  (z >= 80 cm)')
print('=' * 78)
print(f'{"Site":<11} {"Model":<18} {"N":>3} {"RMSE [K]":>9} '
      f'{"Bias [K]":>9} {"MAE [K]":>8} {"R²":>8}')
print('─' * 78)
for r in agg_rows:
    print(f'{r["site"]:<11} {r["model"]:<18} {r["N"]:>3d} '
          f'{r["RMSE"]:>9.3f} {r["bias"]:>+9.3f} {r["MAE"]:>8.3f} {r["R2"]:>8.4f}')
print('─' * 78)
crit_pass = all(r['RMSE'] <= 2.0 for r in agg_rows)
print(f'Phase-1 success criterion (RMSE ≤ 2 K, all rows):  '
      f'{"PASS" if crit_pass else "FAIL"}')

# ---- per-sensor table ------------------------------------------------------
sensor_rows = []
print()
print('TABLE 2 — Per-sensor residuals  (model_mean − T_eq)')
print('=' * 84)
print(f'{"Site":<5} {"Sensor":<8} {"z[cm]":>5} {"T_obs[K]":>8} '
      f'{"T_H[K]":>8} {"T_D[K]":>8} {"dH":>7} {"dD":>7} {"valid":>6}')
print('─' * 84)
for tag, cfg in SITES.items():
    bundle, st = hfe[tag], stats[tag]
    for i, s in enumerate(bundle['sensors']):
        T_h = float(np.interp(s['depth_cm'], z_cm, st['T_mean_hayne']))
        T_d = float(np.interp(s['depth_cm'], z_cm, st['T_mean_disc']))
        valid = 'Y' if bundle['deep_mask'][i] else '·'
        sensor_rows.append({
            'site': cfg['label'], 'sensor': s['sensor'],
            'depth_cm': s['depth_cm'], 'T_obs': s['T_eq'],
            'T_hayne': T_h, 'T_disc': T_d,
            'res_hayne': T_h - s['T_eq'],
            'res_disc':  T_d - s['T_eq'],
            'valid': valid,
        })
        print(f'{cfg["label"][-2:]:<5} {s["sensor"]:<8} {s["depth_cm"]:>5.0f} '
              f'{s["T_eq"]:>8.2f} {T_h:>8.2f} {T_d:>8.2f} '
              f'{T_h - s["T_eq"]:>+7.2f} {T_d - s["T_eq"]:>+7.2f} {valid:>6}')
print('─' * 84)

# ---- CSV export ------------------------------------------------------------
csv_path = os.path.join(OUT_DIR, 'apollo_validation_stats.csv')
with open(csv_path, 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=list(agg_rows[0].keys()))
    w.writeheader(); w.writerows(agg_rows)
sens_csv = os.path.join(OUT_DIR, 'apollo_validation_per_sensor.csv')
with open(sens_csv, 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=list(sensor_rows[0].keys()))
    w.writeheader(); w.writerows(sensor_rows)
print(f'\nCSV → {csv_path}')
print(f'CSV → {sens_csv}')

# ---- LaTeX booktabs block --------------------------------------------------
latex = textwrap.dedent(r'''
    \begin{table}[h]
    \centering
    \caption{Apollo 15 \& 17 HFE deep-sensor ($z \geq 80$\,cm)
             validation statistics for the Hayne (2017) reference model
             and the Discrete 3-layer model. $N$ is the number of
             sensors entering the metric.}
    \label{tab:apollo_validation}
    \begin{tabular}{llrrrrr}
    \toprule
    Site & Model & $N$ & RMSE [K] & Bias [K] & MAE [K] & $R^{2}$ \\
    \midrule
    ''').strip('\n')
for r in agg_rows:
    latex += (f"\n    {r['site']} & {r['model']} & {r['N']} & "
              f"{r['RMSE']:.2f} & {r['bias']:+.2f} & "
              f"{r['MAE']:.2f} & {r['R2']:.3f} \\\\")
latex += '\n    \\bottomrule\n    \\end{tabular}\n\\end{table}'
print('\nLaTeX (paste into manuscript):')
print(latex)


In [ ]:
# Fig 5: Apollo observed vs modelled scatter (1:1 diagnostic) ─────────────
# All sensors from both sites overlaid; deep sensors (used in RMSE) are
# filled, shallow sensors (borestem-contaminated) are open and faded.

fig, ax = plt.subplots(figsize=(6.0, 6.0), constrained_layout=True)

handles_legend = []
for tag, cfg in SITES.items():
    bundle, st = hfe[tag], stats[tag]
    deep = bundle['deep_mask']
    T_obs = bundle['T_eq_all']
    T_h_at = np.interp(bundle['depth_cm_all'], z_cm, st['T_mean_hayne'])
    T_d_at = np.interp(bundle['depth_cm_all'], z_cm, st['T_mean_disc'])
    site_marker = 'o' if tag == 'A15' else '^'

    # Hayne
    ax.scatter(T_obs[deep],  T_h_at[deep],  s=70, marker=site_marker,
               facecolor=CLR_HAYNE, edgecolor='k', lw=0.6, zorder=4,
               label=f'{cfg["label"]} · Hayne (deep)')
    ax.scatter(T_obs[~deep], T_h_at[~deep], s=55, marker=site_marker,
               facecolor='none', edgecolor=CLR_HAYNE, lw=1.2, alpha=0.55,
               zorder=3, label=f'{cfg["label"]} · Hayne (excl.)')
    # Discrete
    ax.scatter(T_obs[deep],  T_d_at[deep],  s=70, marker=site_marker,
               facecolor=CLR_DISC, edgecolor='k', lw=0.6, zorder=4,
               label=f'{cfg["label"]} · Discrete (deep)')
    ax.scatter(T_obs[~deep], T_d_at[~deep], s=55, marker=site_marker,
               facecolor='none', edgecolor=CLR_DISC, lw=1.2, alpha=0.55,
               zorder=3, label=f'{cfg["label"]} · Discrete (excl.)')

# 1:1 line and ±1 K / ±2 K bands
T_lo, T_hi = 200.0, 270.0
xs = np.linspace(T_lo, T_hi, 50)
ax.fill_between(xs, xs - 2, xs + 2, color='#D5DBDB', alpha=0.45,
                zorder=1, label='±2 K')
ax.fill_between(xs, xs - 1, xs + 1, color='#A9DFBF', alpha=0.55,
                zorder=2, label='±1 K')
ax.plot(xs, xs, color='k', lw=1.2, ls='--', label='1:1', zorder=5)

ax.set_xlim(T_lo, T_hi); ax.set_ylim(T_lo, T_hi)
ax.set_aspect('equal')
ax.set_xlabel('Apollo HFE  T_obs  [K]', fontsize=11)
ax.set_ylabel('Model  T_model  [K]', fontsize=11)
ax.set_title('Fig 5 — Modelled vs observed equilibrium temperature\n'
             '(circles = A15, triangles = A17; filled = deep / used in RMSE)',
             fontsize=11, weight='bold')
ax.grid(alpha=0.2)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0,
          fontsize=8, frameon=True)
save_figure(fig, 'apollo_obs_vs_model_scatter', output_dir=OUT_DIR)
plt.show()


In [ ]:
# Fig 6: Residual vs depth (model - obs) ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharey=True,
                         constrained_layout=True)
for ax, (tag, cfg) in zip(axes, SITES.items()):
    bundle, st = hfe[tag], stats[tag]
    deep = bundle['deep_mask']
    z_obs = bundle['depth_cm_all']
    res_h = np.interp(z_obs, z_cm, st['T_mean_hayne']) - bundle['T_eq_all']
    res_d = np.interp(z_obs, z_cm, st['T_mean_disc'])  - bundle['T_eq_all']

    # ±2 K and ±1 K target bands
    ax.axvspan(-2, 2, color='#D5DBDB', alpha=0.45, label='±2 K target')
    ax.axvspan(-1, 1, color='#A9DFBF', alpha=0.55, label='±1 K target')
    ax.axvline(0, color='k', lw=1.0, ls='-')
    ax.axhline(cfg['MIN_DEPTH_CM'], color=ZONE_LINE, ls='--', lw=1.0, alpha=0.7)
    ax.axhspan(0, cfg['MIN_DEPTH_CM'], color=ZONE_CLR, alpha=0.45)

    # Plot residuals
    for i, s in enumerate(bundle['sensors']):
        mk = 'o' if s['stype'] == 'TG' else 's'
        a  = 1.0 if deep[i] else 0.4
        ax.scatter(res_h[i], z_obs[i], marker=mk, s=80,
                   facecolor=CLR_HAYNE, edgecolor='k', lw=0.5, alpha=a, zorder=4)
        ax.scatter(res_d[i], z_obs[i], marker=mk, s=80,
                   facecolor=CLR_DISC,  edgecolor='k', lw=0.5, alpha=a, zorder=4)
        ax.plot([0, res_h[i]], [z_obs[i], z_obs[i]], color=CLR_HAYNE,
                alpha=a*0.6, lw=1.2, zorder=3)
        ax.plot([0, res_d[i]], [z_obs[i], z_obs[i]], color=CLR_DISC,
                alpha=a*0.6, lw=1.2, zorder=3)

    ax.invert_yaxis()
    ax.set_ylim(cfg['y_lim'], 0)
    ax.set_xlim(-8, 8)
    ax.set_xlabel('Residual  T_model − T_obs  [K]', fontsize=10)
    if tag == 'A15':
        ax.set_ylabel('Depth  [cm]', fontsize=10)
    ax.set_title(f'{cfg["label"]}', fontsize=11, weight='bold')
    ax.grid(alpha=0.2)

# Shared legend
from matplotlib.patches import Patch as _Patch
legend_handles = [
    Line2D([],[], marker='o', ls='none', color=CLR_HAYNE, mec='k',
           label='Hayne 2017'),
    Line2D([],[], marker='o', ls='none', color=CLR_DISC, mec='k',
           label='Discrete 3-layer'),
    _Patch(facecolor='#A9DFBF', edgecolor='none', alpha=0.55, label='±1 K target'),
    _Patch(facecolor='#D5DBDB', edgecolor='none', alpha=0.45, label='±2 K target'),
    _Patch(facecolor=ZONE_CLR,  edgecolor=ZONE_LINE, alpha=0.45,
           label='Borestem zone (excl.)'),
]
fig.legend(handles=legend_handles, loc='lower center',
           bbox_to_anchor=(0.5, -0.04), ncol=5, fontsize=9)
fig.suptitle('Fig 6 — Residual T(model) − T(obs) versus depth',
             fontsize=11.5, weight='bold')
save_figure(fig, 'apollo_residual_vs_depth', output_dir=OUT_DIR)
plt.show()


In [ ]:
# Fig 7: Diurnal amplitude vs depth — the borestem signature ───────────
# Amplitude := half the peak-to-peak swing of the modelled and observed
# temperature at each sensor depth.  On a semi-log axis the regolith
# diffusion solution falls off linearly with slope ≈ 1 / skin-depth.
# A flat (or rising) tail below 80 cm reveals the borestem heat-short.

from lunar.constants import LUNATION_SECONDS as _T_lun

def _phase_fold_amp(lst_arr, T_arr, n_bins=72):
    edges = np.linspace(0, 24, n_bins + 1)
    med = np.full(n_bins, np.nan)
    for k in range(n_bins):
        sel = (lst_arr >= edges[k]) & (lst_arr < edges[k+1])
        if sel.sum() >= 5:
            med[k] = np.median(T_arr[sel])
    if not np.any(np.isfinite(med)):
        return np.nan
    return 0.5 * (np.nanmax(med) - np.nanmin(med))

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharey=True,
                         constrained_layout=True)

for ax, (tag, cfg) in zip(axes, SITES.items()):
    bundle = hfe[tag]
    out_h  = runs[tag]['out_hayne']
    out_d  = runs[tag]['out_disc']

    # Modelled amplitude at every numeric layer
    A_h_model = 0.5 * (out_h.T.max(axis=1) - out_h.T.min(axis=1))
    A_d_model = 0.5 * (out_d.T.max(axis=1) - out_d.T.min(axis=1))
    ax.semilogx(np.maximum(A_h_model, 1e-3), z_cm, color=CLR_HAYNE,
                lw=2.0, ls='--', label='Hayne 2017 (model)')
    ax.semilogx(np.maximum(A_d_model, 1e-3), z_cm, color=CLR_DISC,
                lw=2.0, ls='-',  label='Discrete (model)')

    # Observed amplitudes per sensor
    for s in bundle['sensors']:
        sensor = s['sensor']
        depth  = s['depth_cm']
        # gather all valid observation rows for this sensor across both probes
        lst_full, T_full = [], []
        for dtab in (bundle['d1'], bundle['d2']):
            mask = dtab['sensor'] == sensor
            if not mask.any():
                continue
            t_u = iso_to_seconds(dtab['time_iso'][mask])
            T_v = dtab['T'][mask].astype(float)
            ok  = np.isfinite(T_v) & (T_v > 50.0)
            t_u, T_v = t_u[ok], T_v[ok]
            if t_u.size < 50:
                continue
            lst_full.append(spice_lst(t_u, cfg['lon']))
            T_full.append(T_v)
        if not lst_full:
            continue
        lst_full = np.concatenate(lst_full)
        T_full   = np.concatenate(T_full)
        amp = _phase_fold_amp(lst_full, T_full)
        if not np.isfinite(amp):
            continue
        is_deep = depth >= cfg['MIN_DEPTH_CM']
        clr  = CLR_TG if s['stype'] == 'TG' else CLR_TR
        mfc  = clr if is_deep else 'none'
        a    = 1.0 if is_deep else 0.85
        ax.scatter(max(amp, 1e-3), depth, marker='o', s=70,
                   facecolor=mfc, edgecolor=clr, lw=1.4, alpha=a, zorder=4)

    # Skin-depth annotation: ~3–5 cm
    ax.axhspan(0, 5, color='#FCF3CF', alpha=0.55, zorder=0)
    ax.axhline(cfg['MIN_DEPTH_CM'], color=ZONE_LINE, lw=1.0, ls='--',
               alpha=0.7)
    ax.axhspan(0, cfg['MIN_DEPTH_CM'], color=ZONE_CLR, alpha=0.30, zorder=0)
    ax.text(0.03, 4, 'skin depth\n(3–5 cm)', fontsize=8, va='top', ha='left',
            transform=ax.get_yaxis_transform(), color='#7E5109')

    ax.invert_yaxis()
    ax.set_ylim(cfg['y_lim'], 0)
    ax.set_xlim(1e-2, 200)
    ax.set_xlabel('Diurnal amplitude  ½(T_max − T_min)  [K]', fontsize=10)
    if tag == 'A15':
        ax.set_ylabel('Depth  [cm]', fontsize=10)
    ax.set_title(f'{cfg["label"]}', fontsize=11, weight='bold')
    ax.grid(which='both', alpha=0.2)

handles = [
    Line2D([],[], color=CLR_HAYNE, lw=2.0, ls='--', label='Hayne 2017 (model)'),
    Line2D([],[], color=CLR_DISC, lw=2.0, ls='-',  label='Discrete (model)'),
    Line2D([],[], marker='o', color=CLR_TG, ls='none', mec=CLR_TG,
           label='Apollo TG (deep filled)'),
    Line2D([],[], marker='o', color=CLR_TR, ls='none', mec=CLR_TR,
           label='Apollo TR (deep filled)'),
]
fig.legend(handles=handles, loc='lower center',
           bbox_to_anchor=(0.5, -0.04), ncol=4, fontsize=9)
fig.suptitle('Fig 7 — Diurnal amplitude vs depth (skin-depth diagnostic;'
             ' borestem heat-short produces the > 1 K tail at 30–60 cm)',
             fontsize=11, weight='bold')
save_figure(fig, 'apollo_amplitude_vs_depth', output_dir=OUT_DIR)
plt.show()


In [ ]:
# Fig 8: Spin-up convergence diagnostic ─────────────────────────────────
# We re-run a short version of the solver capturing T(depth) at the end of
# every lunation, then plot |dT/dlunation| vs lunation count for two
# representative depths (5 cm, 1 m) at each site.  This demonstrates that
# the 100-lunation, 0.01 K convergence target is reached well before
# the budget is exhausted.

from lunar.solver import PixelInputs, solve_pixel

N_DIAG = 50          # diagnostic lunations
DT_DIAG = 3600.0
N_T_DIAG = int(T_LUNAR / DT_DIAG) + 1
t_diag = np.linspace(0.0, T_LUNAR, N_T_DIAG)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True,
                         constrained_layout=True)

z_targets_cm = np.array([5.0, 100.0])
ms_per_K = {'5 cm': 'o', '100 cm': 's'}

for ax, (tag, cfg) in zip(axes, SITES.items()):
    cos_lat = np.cos(np.deg2rad(cfg['lat']))
    insol   = (S0 * cos_lat *
               np.maximum(0.0, np.cos(2*np.pi*t_diag/T_LUNAR)))

    # Same Hayne init as the main solver
    K_init = conductivity_hayne(np.full_like(grid.z_mid, cfg['T_MEAN_EFF']),
                                grid.z_mid)
    R_z    = np.cumsum(grid.dz / K_init)
    T0     = cfg['T_MEAN_EFF'] + cfg['Q_BASAL'] * R_z
    T_curr = T0.copy()
    T_at_iter = []   # T at end of each lunation, all depths

    for _ in range(N_DIAG):
        inp = PixelInputs(
            grid=grid, t=t_diag, bc_mode='radiative',
            insolation=insol, albedo=cfg['albedo'],
            emissivity=cfg['emissivity'], Q_b=cfg['Q_BASAL'],
            T_init=T_curr,
            n_lunations_spinup=1, spinup_tol_K=1e-9,  # one cycle, no inner loop
        )
        out = solve_pixel(inp)
        T_curr = out.T[:, -1]                # state at end of lunation
        T_at_iter.append(T_curr.copy())

    T_arr = np.array(T_at_iter)               # (N_DIAG, n_layers)

    for z_cm_t, label in zip(z_targets_cm, ('5 cm', '100 cm')):
        iz   = int(np.argmin(np.abs(z_cm - z_cm_t)))
        dT   = np.abs(np.diff(T_arr[:, iz]))
        ax.semilogy(np.arange(1, len(dT)+1), np.maximum(dT, 1e-6),
                    marker=ms_per_K[label], lw=1.4, ms=4,
                    label=f'{cfg["label"]} · z={label}')
    ax.axhline(SPINUP_TOL, color='red', lw=1.0, ls='--',
               label=f'Tolerance = {SPINUP_TOL} K')
    ax.set_xlabel('Lunation #', fontsize=10)
    if tag == 'A15':
        ax.set_ylabel('|ΔT(z)| between successive lunations  [K]', fontsize=10)
    ax.set_title(f'{cfg["label"]}', fontsize=11, weight='bold')
    ax.grid(which='both', alpha=0.2)
    ax.legend(fontsize=8, loc='upper right')

fig.suptitle('Fig 8 — Spin-up convergence  '
             '(both sites reach 0.01 K target well within 100 lunations)',
             fontsize=11, weight='bold')
save_figure(fig, 'apollo_spinup_convergence', output_dir=OUT_DIR)
plt.show()
